# Algorithm Agent - NOAA Near-surface Atmospheric Properties

In [ ]:
%env AWS_PROFILE=kmnlp
%env ZARR_REFERENCE_PATH=s3://data-c6c22a2e42294c11b52ee7f0c792c071/noaa-cdr-ocean-nearsurface-atmos-profiles-pds/daily/zarr_reference_final.json

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
import os
import fsspec
import numpy as np
import xarray as xr
from dask.distributed import Client, LocalCluster

cluster = LocalCluster()
client = cluster.get_client()
client

In [ ]:
%%time
zarr_reference_path = os.getenv('ZARR_REFERENCE_PATH')
mapper = fsspec.get_mapper(
    'reference://', 
    fo=zarr_reference_path,
    remote_protocol='s3',
)
ds = xr.open_zarr(mapper, consolidated=False)
ds

In [ ]:
wind_speed = ds.wind_speed
wind_speed

---

In [ ]:
from rich import print as rprint
from e84_geoai_common.llm.models import BedrockClaudeLLM, CLAUDE_BEDROCK_MODEL_IDS

from llm_agent.agents.algorithm_agent.algorithm.base import *
from llm_agent.agents.algorithm_agent.algorithm.data_array_functions import *
from llm_agent.agents.algorithm_agent.algorithm.general_plot_functions import *
from llm_agent.agents.algorithm_agent.algorithm.exec import *
from llm_agent.agents.algorithm_agent.algorithm_agent import *
from llm_agent.agents.algorithm_agent.interface import *

In [ ]:
%matplotlib inline

In [ ]:
input_vars = [
    InputVariable(
        name="wind_speed",
        variable_type="DataArray",
        description="""NOAA climate data record of ocean near-surface wind speed. \
Units: m s-1. Temporal resolution: PT3H. There are 8 data points for each day.\
""")
]

---

In [ ]:
llm = BedrockClaudeLLM(model_id=CLAUDE_BEDROCK_MODEL_IDS['Claude 3.5 Sonnet'])

In [ ]:
agent = AlgorithmAgent(
    llm, 
    problem_spec=AlgorithmProblemSpec, 
    solution_spec=AlgorithmSolutionSpec[DataArrayFunctionSet | GeneralPlottingFunctionSet],
)

In [ ]:
rprint(agent.system_prompt)

---

## Problem 1

In [ ]:
problem_spec = AlgorithmProblemSpec(
    statement="What was the mean wind speed in the Black Sea on June 01, 2021.",
    inputs_provided=input_vars,
    outputs_requested=[
        OutputVariable(
            name="wind_speed_mean_scalar", 
            variable_type="scalar_float", 
            description="The mean value of wind speed.",
        ),
    ],
)
rprint(problem_spec)

In [ ]:
solution = agent.run(problem_spec)

In [ ]:
rprint(solution.thoughts)
rprint(solution.pseudocode)
rprint(solution.algorithm)
rprint(solution.explanation)

---

### Execute

In [ ]:
%%time
out = execute_algorithm(solution.algorithm, input_vars={"wind_speed": wind_speed})
out

---
---

## Problem 2

In [ ]:
problem_spec = AlgorithmProblemSpec(
    statement="Plot the wind speed trend in the Black Sea on June 01, 2021.",
    inputs_provided=input_vars,
    outputs_requested=[
        OutputVariable(
            name="wind_speed_trend", 
            variable_type="line_plot", 
            description="The wind speed trend in the Black Sea on June 01, 2021.",
        ),
    ],
)
rprint(problem_spec)

In [ ]:
solution = agent.run(problem_spec)

In [ ]:
rprint(solution.thoughts)
rprint(solution.pseudocode)
rprint(solution.algorithm)
rprint(solution.explanation)

---

### Execute

In [ ]:
%%time
out = execute_algorithm(solution.algorithm, input_vars={"wind_speed": wind_speed})
out

In [ ]:
out["wind_speed_trend"]

## Problem 3

In [ ]:
problem_spec = AlgorithmProblemSpec(
    statement="Find the time and location of the max wind speed value in the Black Sea on June 01, 2021.",
    inputs_provided=input_vars,
    outputs_requested=[
        OutputVariable(
            name="max_wind_speed_time", 
            variable_type="datetime_string", 
            description="Time of the max wind speed value.",
        ),
        OutputVariable(
            name="max_wind_speed_lat", 
            variable_type="latitude_deg_float", 
            description="Lat coord of the max wind speed value.",
        ),
        OutputVariable(
            name="max_wind_speed_lon", 
            variable_type="longitude_deg_float", 
            description="Lon coord of the max wind speed value.",
        ),
        OutputVariable(
            name="max_wind_speed_value", 
            variable_type="scalar_float", 
            description="Max wind speed value.",
        ),
    ],
)
rprint(problem_spec)

In [ ]:
solution = agent.run(problem_spec)

In [ ]:
rprint(solution.thoughts)
rprint(solution.pseudocode)
rprint(solution.algorithm)
rprint(solution.explanation)

---

### Execute

In [ ]:
%%time
out = execute_algorithm(solution.algorithm, input_vars={"wind_speed": wind_speed})
out